In [6]:
import os
import json
from PIL import Image
import numpy as np

folder = r"D:\Desktop_fake\LegaPKMN_v3\grafiche\pokemon"
output_json = r"D:\Desktop_fake\MawileBot\home\utils\pokemon_vectors_9.json"

f = 0.3  # distanza principale

pokemon_vectors = {}

for filename in os.listdir(folder):
    if not filename.lower().endswith(".png"):
        continue

    path = os.path.join(folder, filename)
    name = os.path.splitext(filename)[0]

    if 'mega' in name and ('yanmega' not in name or 'mrmimegalar' not in name):
        continue

    img_rgba = Image.open(path).convert("RGBA")

    # Create white background
    white_bg = Image.new("RGBA", img_rgba.size, (255, 255, 255, 255))

    # Composite sprite over white
    img_flat = Image.alpha_composite(white_bg, img_rgba)

    # Now convert to grayscale
    img = img_flat.convert("L")
    mat = np.array(img)

    h, w = mat.shape
    cx, cy = w // 2, h // 2
    dx, dy = int(w * f), int(h * f)

    coords = [
        (cx, cy),                         # center
        (cx, max(cy - dy, 0)),            # up
        (cx, min(cy + dy, h - 1)),        # down
        (max(cx - dx, 0), cy),            # left
        (min(cx + dx, w - 1), cy),        # right
        (max(cx - dx//2, 0), max(cy - dy//2, 0)),  # up-left
        (min(cx + dx//2, w - 1), max(cy - dy//2, 0)), # up-right
        (max(cx - dx//2, 0), min(cy + dy//2, h - 1)), # down-left
        (min(cx + dx//2, w - 1), min(cy + dy//2, h - 1)), # down-right
    ]

    vector = [int(mat[y, x]) for x, y in coords]
    pokemon_vectors[name] = vector

# Save JSON
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(pokemon_vectors, f, indent=4)

print("Saved 9-point embeddings to:", output_json)


Saved 9-point embeddings to: D:\Desktop_fake\MawileBot\home\utils\pokemon_vectors_9.json


In [7]:
import json
from collections import defaultdict

json_path = r"D:\Desktop_fake\MawileBot\home\utils\pokemon_vectors_9.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# invert: vector -> list of pokemon with that vector
groups = defaultdict(list)

for name, vector in data.items():
    groups[tuple(vector)].append(name)

# print only duplicates
found = False
for vector, names in groups.items():
    if len(names) > 1:
        found = True
        print("Vector:", list(vector))
        print("Pokémon:", names)
        print("-" * 40)

if not found:
    print("No identical embeddings found 🎯")


Vector: [185, 136, 206, 255, 210, 121, 77, 65, 8]
Pokémon: ['aegislash', 'aegislashspada']
----------------------------------------
Vector: [43, 51, 193, 164, 164, 255, 255, 78, 72]
Pokémon: ['appletungiga', 'flapplegiga']
----------------------------------------
Vector: [134, 193, 86, 157, 255, 184, 255, 96, 88]
Pokémon: ['applin', 'applinappletun', 'applindipplin', 'applinflapple']
----------------------------------------
Vector: [99, 201, 133, 109, 255, 187, 255, 143, 136]
Pokémon: ['applinappletunshiny', 'applindipplinshiny', 'applinflappleshiny', 'applinshiny']
----------------------------------------
Vector: [143, 255, 255, 142, 114, 255, 255, 117, 135]
Pokémon: ['basculegion', 'basculegionmaschio']
----------------------------------------
Vector: [139, 255, 255, 231, 13, 255, 255, 132, 255]
Pokémon: ['basculegionbianco', 'basculegionfemmina']
----------------------------------------
Vector: [12, 74, 255, 144, 255, 147, 155, 116, 161]
Pokémon: ['basculin', 'basculinrosso']
------

In [8]:
from PIL import Image, ImageDraw
import numpy as np

# Image path
img_path = r"D:\Desktop_fake\LegaPKMN_v3\grafiche\pokemon\archeokyogre.png"

# Load image
img = Image.open(img_path).convert("RGB")
mat = np.array(img)

h, w = mat.shape[:2]

# Cross parameters
f = 0.3
cx, cy = w // 2, h // 2
dx, dy = int(w * f), int(h * f)

# 9 points coordinates
coords = [
    (cx, cy),                         # center
    (cx, max(cy - dy, 0)),            # up
    (cx, min(cy + dy, h - 1)),        # down
    (max(cx - dx, 0), cy),            # left
    (min(cx + dx, w - 1), cy),        # right
    (max(cx - dx//2, 0), max(cy - dy//2, 0)),  # up-left
    (min(cx + dx//2, w - 1), max(cy - dy//2, 0)), # up-right
    (max(cx - dx//2, 0), min(cy + dy//2, h - 1)), # down-left
    (min(cx + dx//2, w - 1), min(cy + dy//2, h - 1)), # down-right
]

# Draw points
draw = ImageDraw.Draw(img)
radius = 5

for x, y in coords:
    draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=(255, 0, 0))

# Show the image
img.show()
